# NMF on ModulAir 00692

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [8]:
#importing data from Modulair MOD-00692
data = pd.read_csv('MOD-00692.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:12Z,577611775,2025-12-31T18:59:12Z,MOD-00692,49.9,0.2,21.729,1.378,0.226,0.085,0.028,...,34.593,22.724,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.09
2025-12-31T23:58:12Z,577611776,2025-12-31T18:58:12Z,MOD-00692,49.4,0.2,9.733,0.906,0.247,0.116,0.062,...,34.140,23.490,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.61
2025-12-31T23:57:12Z,577609805,2025-12-31T18:57:12Z,MOD-00692,49.5,0.1,7.114,0.735,0.261,0.044,0.039,...,33.655,23.823,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.58
2025-12-31T23:56:12Z,577609806,2025-12-31T18:56:12Z,MOD-00692,49.7,0.1,7.072,0.655,0.242,0.094,0.068,...,33.883,23.447,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.24
2025-12-31T23:55:12Z,577609807,2025-12-31T18:55:12Z,MOD-00692,50.2,0.1,7.851,0.916,0.297,0.113,0.051,...,34.810,23.385,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,3.75


In [9]:
start = data.index.min()
end = data.index.max()

print(start, end)

2025-04-02T16:31:31Z 2025-12-31T23:59:12Z


In [10]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:12Z,2025-12-31T18:59:12Z,774.794,2.896,34.593,22.724,21.729,1.378,0.226,0.085,0.028,0.034
2025-12-31T23:58:12Z,2025-12-31T18:58:12Z,739.435,2.559,34.140,23.490,9.733,0.906,0.247,0.116,0.062,0.013
2025-12-31T23:57:12Z,2025-12-31T18:57:12Z,732.687,2.768,33.655,23.823,7.114,0.735,0.261,0.044,0.039,0.032
2025-12-31T23:56:12Z,2025-12-31T18:56:12Z,739.114,2.896,33.883,23.447,7.072,0.655,0.242,0.094,0.068,0.011
2025-12-31T23:55:12Z,2025-12-31T18:55:12Z,747.379,2.907,34.810,23.385,7.851,0.916,0.297,0.113,0.051,0.016


In [11]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:12Z,774.794,2.896,34.593,22.724,21.729,1.378,0.226,0.085,0.028,0.034
1,2025-12-31T18:58:12Z,739.435,2.559,34.140,23.490,9.733,0.906,0.247,0.116,0.062,0.013
2,2025-12-31T18:57:12Z,732.687,2.768,33.655,23.823,7.114,0.735,0.261,0.044,0.039,0.032
3,2025-12-31T18:56:12Z,739.114,2.896,33.883,23.447,7.072,0.655,0.242,0.094,0.068,0.011
4,2025-12-31T18:55:12Z,747.379,2.907,34.810,23.385,7.851,0.916,0.297,0.113,0.051,0.016


In [12]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:12,774.794,2.896,34.593,22.724,21.729,1.378,0.226,0.085,0.028,0.034
1,2025-12-31 18:58:12,739.435,2.559,34.140,23.490,9.733,0.906,0.247,0.116,0.062,0.013
2,2025-12-31 18:57:12,732.687,2.768,33.655,23.823,7.114,0.735,0.261,0.044,0.039,0.032
3,2025-12-31 18:56:12,739.114,2.896,33.883,23.447,7.072,0.655,0.242,0.094,0.068,0.011
4,2025-12-31 18:55:12,747.379,2.907,34.810,23.385,7.851,0.916,0.297,0.113,0.051,0.016


In [13]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
1,2025-04-02 13:00:00,1800.74,8.50,66.59,2.89,2.46,0.33,0.13,0.03,0.05,0.03
4,2025-04-15 12:00:00,1453.26,23.11,73.87,2.52,7.48,0.73,0.25,0.07,0.08,0.06
5,2025-04-15 13:00:00,1140.28,20.30,69.41,2.59,4.60,0.45,0.17,0.05,0.08,0.05
6,2025-04-15 14:00:00,901.10,8.81,64.97,2.46,2.56,0.30,0.14,0.05,0.06,0.05
7,2025-04-15 15:00:00,819.78,8.92,63.85,2.35,1.60,0.24,0.12,0.04,0.06,0.05
...,...,...,...,...,...,...,...,...,...,...,...
6192,2025-12-31 14:00:00,679.63,32.58,28.95,1.89,8.78,0.64,0.19,0.04,0.03,0.02
6193,2025-12-31 15:00:00,690.96,32.87,28.35,1.93,7.56,0.62,0.19,0.05,0.04,0.02
6194,2025-12-31 16:00:00,705.10,33.32,27.32,2.35,8.51,0.74,0.22,0.05,0.04,0.02
6195,2025-12-31 17:00:00,762.18,34.65,23.12,2.10,9.62,0.98,0.31,0.08,0.06,0.02


In [14]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-02 13:00:00,1800.74,8.50,66.59,2.89,2.46,0.33,0.13,0.03,0.05,0.03
2025-04-15 12:00:00,1453.26,23.11,73.87,2.52,7.48,0.73,0.25,0.07,0.08,0.06
2025-04-15 13:00:00,1140.28,20.30,69.41,2.59,4.60,0.45,0.17,0.05,0.08,0.05
2025-04-15 14:00:00,901.10,8.81,64.97,2.46,2.56,0.30,0.14,0.05,0.06,0.05
2025-04-15 15:00:00,819.78,8.92,63.85,2.35,1.60,0.24,0.12,0.04,0.06,0.05


In [15]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [16]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [17]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-02 13:00:00,0.936866,0.184863,0.779742,0.052863,0.031742,0.007121,0.007096,0.006356,0.012469,0.022222
2025-04-15 12:00:00,0.756083,0.502610,0.864988,0.046095,0.096516,0.015753,0.013646,0.014831,0.019950,0.044444
2025-04-15 13:00:00,0.593250,0.441496,0.812763,0.047375,0.059355,0.009711,0.009279,0.010593,0.019950,0.037037
2025-04-15 14:00:00,0.468813,0.191605,0.760773,0.044997,0.033032,0.006474,0.007642,0.010593,0.014963,0.037037
2025-04-15 15:00:00,0.426504,0.193997,0.747658,0.042985,0.020645,0.005179,0.006550,0.008475,0.014963,0.037037


In [18]:
data.to_csv(r'MOD-000692_timeseries_hourly_scaled.csv')

In [19]:
len(data)

6094

## Implementing NMF

In [20]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [21]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-02 13:00:00,0.556496,0.308978,0.928050,0.061349,0.185315,0.042580,0.028963,0.024647,0.032874,0.071748
2025-04-15 12:00:00,0.614914,0.548004,0.919076,0.068227,0.152481,0.032171,0.021217,0.017657,0.026794,0.064730
2025-04-15 13:00:00,0.525009,0.463368,0.838525,0.059636,0.087399,0.014300,0.011073,0.010228,0.020136,0.053408
2025-04-15 14:00:00,0.408438,0.211238,0.783981,0.047565,0.058774,0.007649,0.008903,0.009791,0.019720,0.050140
2025-04-15 15:00:00,0.392357,0.205311,0.759604,0.045891,0.050264,0.004682,0.006449,0.007442,0.017244,0.046197
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.395883,0.694427,0.321728,0.042074,0.093813,0.019848,0.007424,0.002681,0.004692,0.018784
2025-12-31 15:00:00,0.392757,0.703742,0.318295,0.041970,0.082657,0.017304,0.006472,0.002338,0.004661,0.018688
2025-12-31 16:00:00,0.397800,0.714552,0.307550,0.042124,0.095923,0.020675,0.007733,0.002793,0.004536,0.018234


In [22]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.027555,0.178635,0.031709,0.016109
1,0.052127,0.163367,0.024668,0.011348
2,0.045079,0.150400,0.009212,0.007073
3,0.020516,0.152409,0.001744,0.007466
4,0.020142,0.147526,0.000000,0.005805
...,...,...,...,...
6089,0.067545,0.030314,0.021271,0.000000
6090,0.068677,0.028980,0.018544,0.000000
6091,0.069483,0.026299,0.022157,0.000000
6092,0.072398,0.016222,0.028875,0.003352


In [23]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [24]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.331698,10.043793,2.612404,0.476742,0.000000,0.000000,0.000000,0.000000,0.042695,0.178120
Feature 2,2.042201,0.000000,4.792272,0.243547,0.340714,0.000000,0.000000,0.000000,0.059659,0.222752
Feature 3,1.945857,0.752923,0.000000,0.117050,3.924859,0.933110,0.349023,0.126064,0.000000,0.000000
Feature 4,0.659777,0.518287,0.000000,0.061767,0.000000,0.806551,1.110968,1.281924,1.306144,1.679124


In [25]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-04-02 13:00:00,0.027555,0.178635,0.031709,0.016109
2025-04-15 12:00:00,0.052127,0.163367,0.024668,0.011348
2025-04-15 13:00:00,0.045079,0.150400,0.009212,0.007073
2025-04-15 14:00:00,0.020516,0.152409,0.001744,0.007466
2025-04-15 15:00:00,0.020142,0.147526,0.000000,0.005805
...,...,...,...,...
2025-12-31 14:00:00,0.067545,0.030314,0.021271,0.000000
2025-12-31 15:00:00,0.068677,0.028980,0.018544,0.000000
2025-12-31 16:00:00,0.069483,0.026299,0.022157,0.000000


In [26]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.331698,10.043793,2.612404,0.476742,0.000000,0.000000,0.000000,0.000000,0.042695,0.178120
Factor 2,2.042201,0.000000,4.792272,0.243547,0.340714,0.000000,0.000000,0.000000,0.059659,0.222752
Factor 3,1.945857,0.752923,0.000000,0.117050,3.924859,0.933110,0.349023,0.126064,0.000000,0.000000
Factor 4,0.659777,0.518287,0.000000,0.061767,0.000000,0.806551,1.110968,1.281924,1.306144,1.679124


In [27]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.523759,0.297817,0.140410,0.025334,0.012680
no,0.546109,0.336478,0.080017,0.022469,0.014927
no2,0.945531,0.000000,0.042300,0.015494,-0.003326
o3,0.312933,0.692357,0.000000,0.000000,-0.005290
bin0,0.000000,0.151536,0.863746,0.000000,-0.015282
bin1,0.000000,0.000000,0.820116,0.377214,-0.197330
bin2,0.000000,0.000000,0.428908,0.726483,-0.155391
bin3,0.000000,0.000000,0.163183,0.882995,-0.046178
bin4,0.081874,0.137982,0.000000,0.795403,-0.015259
bin5,0.181449,0.273680,0.000000,0.543193,0.001678


In [28]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')